In [ ]:
# %pip install seleniumbase

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
# from selenium_stealth import stealth
from seleniumbase import DriverContext, SB
import re
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# 1. 드라이버 컨텍스트 생성 (get_driver()를 쓰지 말고 __enter__()를 씁니다)
context = DriverContext(uc=True)
sb = context.__enter__()

# 2. URL로 이동
url = "https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000222833&t_page=%ED%86%B5%ED%95%A9%EA%B2%80%EC%83%89%EA%B2%B0%EA%B3%BC%ED%8E%98%EC%9D%B4%EC%A7%80&t_click=%EA%B2%80%EC%83%89%EC%83%81%ED%92%88%EC%83%81%EC%84%B8&t_search_name=%EC%97%90%EC%8A%A4%ED%8A%B8%EB%9D%BC&t_number=1&dispCatNo=1000001000100150001%2C1000001000800130001&trackingCd=Result_1&tab=qna"
sb.uc_open_with_reconnect(url, reconnect_time=10)

# 3. 캡차 대응
sb.uc_gui_handle_captcha()

[올영어워즈1등 크림] 에스트라 아토베리어365 크림 80ml 기획 (+하이드로 에센스25ml+세라-히알 앰플7ml) | 올리브영


In [23]:
# QnA 버튼 클릭
qna_button = sb.find_element(By.XPATH, '//*[@id="main"]/div[2]/div/div[3]/div[2]/div[1]/div/div/button[3]')
qna_button.click()

In [35]:
#페이지 끝까지 스크롤
prev_height = sb.execute_script("return document.body.scrollHeight") # 총 스크롤 길이 변수에 할당
while True:
    # 스크롤 내리기  # 0에서 부터 제일 긴 스크롤까지 해줘
    sb.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.5) # 2초 대기 (stale 에러 방지)

    # 스크롤이 변경됐다면? 새로운 높이 가져오기
    new_height = sb.execute_script("return document.body.scrollHeight") # 총 스크롤 길이
    if new_height == prev_height: # 이전과 같다면 
        break # 멈춤
    prev_height = new_height # 새 스크롤길이로 다시 할당 해줌

In [36]:
# 리뷰 리스트
final_list = []

review_li_list = sb.find_elements(By.XPATH, '//*[@id="tab-panels"]/div/section/div[2]/ul/li')
for review_li in review_li_list:
    question = review_li.find_element(By.XPATH, './div[1]/button/p').text
    answer = review_li.find_element(By.XPATH, './div[2]/p').get_attribute("textContent")
    final_list.append([question, answer])
print("현재까지 수집된 리뷰 수: ", len(final_list))

현재까지 수집된 리뷰 수:  152


In [37]:
print(len(final_list))
print(list(final_list)[-1])

152
['잘 쓰다가 언제부턴가 유분기가 너무 많이 느껴지고 얼굴에 모낭염 좁쌀같은 트러블이 계속 올라오던데,, 리뉴얼이 언제 진행된 건가요? 시기를 알고 싶습니다', '안녕하세요. 에스트라 담당자 입니다.\n\n네 고객님, 에스트라 아토베리어365 크림은 24년 1월 이후 리뉴얼되어 2세대로 운영되고 있는 제품입니다.1세대 제품 대비, 민감 피부에 대한 집요한 연구를 통해 민감 피부에 나타나는 장벽 빈틈을 채워주는 민감 특화 세라마이드를 더해 장벽 보습 효능을 업그레이드하고, 인체적용시험과 안전성테스트를 보다 강화해 민감 피부가 보다 편안하게 사용할 수 있는 제품으로 업그레이드 하였습니다. \n제품 사용후 잘 맞지 않으셨다니 속상하고 안타까운 마음입니다. 고객님께서 구매해주셔서 사용하신 상품이 잘 맞지 않아 사용하시기 어려우신경우 구매처 통해 1차로 문의해보시거나, 구매처에 문의/처리가 어려우신경우 아모레퍼시픽 고객상담실에 유선상 문의주시어 상담을 통해 처리/안내받아보실 수 있습니다. 아래 아모레퍼시픽 고객상담실 번호 남겨드립니다. 아모레퍼시픽 고객상담실: 080-023-5454 \n\n감사합니다.']


In [38]:
df = pd.DataFrame(final_list, columns=['질문', '답변'])
df.to_excel('oy_qna_result.xlsx', index=False)
print(f"총 {len(df)}개 QnA 저장 완료")

총 152개 QnA 저장 완료
